
# 02 — Data Integration and Cleaning

## Objectives
- Load the three yearly Price Paid datasets.
- Standardise transaction IDs.
- Integrate the UPRN lookup dataset.
- Measure the transaction-to-UPRN matching rate.
- Identify suitable English regions for modelling.
- Save the integrated dataset for further processing.

## Important limitation
A UPRN match does not guarantee an EPC match. EPC attributes will be integrated only if suitable records can be obtained.


In [1]:

from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data/raw")
OUTPUT_DIR = Path("../data/interim")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PP_COLUMNS = [
    "transaction_id", "price", "date", "postcode",
    "property_type", "old_new", "duration", "paon",
    "saon", "street", "locality", "town",
    "district", "county", "ppd_category", "record_status"
]

# Load and combine the yearly files
frames = []

for year in [2023, 2024, 2025]:
    df = pd.read_csv(
        DATA_DIR / f"pp-{year}.csv",
        header=None,
        names=PP_COLUMNS,
        dtype={"transaction_id": "string"},
        low_memory=False
    )
    frames.append(df)

sales = pd.concat(frames, ignore_index=True)

sales["transaction_id"] = (
    sales["transaction_id"]
    .str.strip()
    .str.strip("{}")
    .str.upper()
)

sales["price"] = pd.to_numeric(sales["price"], errors="coerce")
sales["date"] = pd.to_datetime(sales["date"], errors="coerce")

print("Combined sales:", len(sales))
print("Unique transaction IDs:", sales["transaction_id"].nunique())

# Load UPRN lookup
lookup = pd.read_csv(
    DATA_DIR / "pp-uprn-lookup-aug-2026.csv",
    header=None,
    names=["transaction_id", "uprn"],
    dtype="string",
    keep_default_na=False
)

lookup["transaction_id"] = (
    lookup["transaction_id"]
    .str.strip()
    .str.strip("{}")
    .str.upper()
)
lookup["uprn"] = lookup["uprn"].str.strip()

lookup = lookup[
    lookup["transaction_id"].ne("") &
    lookup["uprn"].ne("")
].copy()

print("Lookup records:", len(lookup))
print(
    "Duplicate lookup transaction IDs:",
    lookup["transaction_id"].duplicated().sum()
)

# Check duplicate keys before joining
if lookup["transaction_id"].duplicated().any():
    raise ValueError(
        "Lookup has duplicate transaction IDs. Inspect these before joining."
    )

# Join lookup to sales
merged = sales.merge(
    lookup,
    on="transaction_id",
    how="left",
    indicator=True,
    validate="one_to_one"
)

matched = merged["_merge"].eq("both")

print("\n--- MATCH RESULTS ---")
print("Matched transactions:", int(matched.sum()))
print("Unmatched transactions:", int((~matched).sum()))
print(f"Match rate: {matched.mean() * 100:.2f}%")
print(
    "Unique matched UPRNs:",
    merged.loc[matched, "uprn"].nunique()
)

# Save integrated records, retaining unmatched sales too
merged = merged.drop(columns="_merge")
merged.to_csv(OUTPUT_DIR / "sales_with_uprn.csv", index=False)

print("\nSaved:", OUTPUT_DIR / "sales_with_uprn.csv")


Combined sales: 2767042
Unique transaction IDs: 2767042
Lookup records: 84149
Duplicate lookup transaction IDs: 0

--- MATCH RESULTS ---
Matched transactions: 20643
Unmatched transactions: 2746399
Match rate: 0.75%
Unique matched UPRNs: 20499

Saved: ..\data\interim\sales_with_uprn.csv


In [2]:

# Find which counties have the most sales and UPRN matches
region_summary = (
    merged.assign(has_uprn=merged["uprn"].notna())
    .groupby("county", dropna=False)
    .agg(
        transactions=("transaction_id", "size"),
        with_uprn=("has_uprn", "sum")
    )
)

region_summary["match_rate_pct"] = (
    region_summary["with_uprn"] /
    region_summary["transactions"] * 100
)

region_summary = region_summary.sort_values(
    "transactions", ascending=False
)

display(region_summary.head(20).round(2))


,transactions,with_uprn,match_rate_pct
county,,,
GREATER LONDON,311469,1876,0.60
GREATER MANCHESTER,128670,921,0.72
WEST YORKSHIRE,107785,706,0.66
WEST MIDLANDS,103129,751,0.73
KENT,77333,705,0.91
ESSEX,74232,653,0.88
LANCASHIRE,69238,626,0.90
MERSEYSIDE,67156,434,0.65
HAMPSHIRE,66402,492,0.74


In [3]:

# Diagnostic: investigate UPRN lookup coverage

lookup_ids = set(lookup["transaction_id"].dropna())

sales["year"] = sales["date"].dt.year
sales["has_lookup_match"] = sales["transaction_id"].isin(lookup_ids)

print("LOOKUP DATE COVERAGE")
print("Earliest transaction date:", sales.loc[
    sales["has_lookup_match"], "date"
].min())
print("Latest transaction date:", sales.loc[
    sales["has_lookup_match"], "date"
].max())

print("\nMATCH RATE BY YEAR")
yearly_match = sales.groupby("year").agg(
    transactions=("transaction_id", "size"),
    matched=("has_lookup_match", "sum")
)
yearly_match["match_rate_pct"] = (
    yearly_match["matched"] / yearly_match["transactions"] * 100
)
display(yearly_match.round(2))

print("\nSAMPLE TRANSACTION IDS")
print("Sales:", sales["transaction_id"].dropna().head(5).tolist())
print("Lookup:", lookup["transaction_id"].dropna().head(5).tolist())

print("\nLOOKUP UPRN QUALITY")
print("Unique UPRNs:", lookup["uprn"].nunique())
print("Duplicate UPRNs:", lookup["uprn"].duplicated().sum())


LOOKUP DATE COVERAGE
Earliest transaction date: 2023-01-03 00:00:00
Latest transaction date: 2025-12-30 00:00:00

MATCH RATE BY YEAR


,transactions,matched,match_rate_pct
year,,,
2023,861502,343,0.04
2024,931453,1051,0.11
2025,974087,19249,1.98



SAMPLE TRANSACTION IDS
Sales: ['01EB45F0-6229-40F3-E063-4704A8C05FDE', '01EB45F0-622A-40F3-E063-4704A8C05FDE', '01EB45F0-622B-40F3-E063-4704A8C05FDE', '01EB45F0-622C-40F3-E063-4704A8C05FDE', '01EB45F0-622D-40F3-E063-4704A8C05FDE']
Lookup: ['326EFAEB-0447-475E-8A3A-02EE17C340BE', '21205A97-62A5-4A6F-9726-2615DF32050E', '2FABC25E-9857-44F5-974F-1F832707C81C', '09BA97FA-4281-48AC-83BF-1F8FED9B69C3', '258C1482-5E38-41C6-977A-2E40873F7DA0']

LOOKUP UPRN QUALITY
Unique UPRNs: 82998
Duplicate UPRNs: 1151


In [4]:
# Compare counties using transaction volume only
county_summary = (
    sales.groupby("county")
    .agg(
        transactions=("transaction_id", "size"),
        median_price=("price", "median")
    )
    .sort_values("transactions", ascending=False)
)

display(county_summary.head(15).round(2))

,transactions,median_price
county,,
GREATER LONDON,311469,510000.0
GREATER MANCHESTER,128670,220000.0
WEST YORKSHIRE,107785,197000.0
WEST MIDLANDS,103129,223000.0
KENT,77333,340000.0
ESSEX,74232,364950.0
LANCASHIRE,69238,186000.0
MERSEYSIDE,67156,180000.0
HAMPSHIRE,66402,365000.0


In [5]:

# Filter Greater London and perform initial quality checks

london = sales[
    sales["county"].str.strip().str.upper() == "GREATER LONDON"
].copy()

# Basic data quality checks
print("Greater London transactions:", len(london))
print("Date range:", london["date"].min(), "to", london["date"].max())
print("Missing prices:", london["price"].isna().sum())
print("Non-positive prices:", (london["price"] <= 0).sum())
print("Missing postcodes:", london["postcode"].isna().sum())
print("Duplicate transaction IDs:", london["transaction_id"].duplicated().sum())

print("\nTransactions by year:")
display(london.groupby(london["date"].dt.year).size().rename("transactions"))

print("\nProperty types:")
display(london["property_type"].value_counts(dropna=False))

print("\nPrice distribution:")
display(london["price"].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]))

# Save an initial regional extract
from pathlib import Path

output_dir = Path("../data/interim")
output_dir.mkdir(parents=True, exist_ok=True)
london.to_csv(output_dir / "greater_london_raw.csv", index=False)

print("\nSaved to data/interim/greater_london_raw.csv")


Greater London transactions: 311469
Date range: 2023-01-01 00:00:00 to 2025-12-31 00:00:00
Missing prices: 0
Non-positive prices: 0
Missing postcodes: 782
Duplicate transaction IDs: 0

Transactions by year:


date
2023     96500
2024    106790
2025    108179
Name: transactions, dtype: int64


Property types:


property_type
F    160371
T     78802
S     41540
O     17601
D     13155
Name: count, dtype: int64


Price distribution:


count    3.114690e+05
mean     8.124432e+05
std      2.937181e+06
min      1.000000e+00
1%       5.500000e+04
25%      3.722500e+05
50%      5.100000e+05
75%      7.500000e+05
99%      5.300000e+06
max      7.930200e+08
Name: price, dtype: float64


Saved to data/interim/greater_london_raw.csv


In [6]:

import numpy as np
import pandas as pd
from pathlib import Path

# Work on a copy
df = london.copy()

# Standardise text fields
for col in ["postcode", "property_type", "old_new", "duration", "district"]:
    df[col] = df[col].astype("string").str.strip().str.upper()
    df[col] = df[col].replace("", pd.NA)

# Investigate unusually low and high prices before filtering
print("Prices below £1,000:", (df["price"] < 1000).sum())
print("Prices below £10,000:", (df["price"] < 10000).sum())
print("Prices above £10 million:", (df["price"] > 10_000_000).sum())
print("Prices above £50 million:", (df["price"] > 50_000_000).sum())

print("\nLowest 20 transactions:")
display(
    df.nsmallest(20, "price")[
        ["price", "date", "property_type", "postcode", "district"]
    ]
)

print("\nHighest 20 transactions:")
display(
    df.nlargest(20, "price")[
        ["price", "date", "property_type", "postcode", "district"]
    ]
)

# Remove records with invalid target values.
# Flag unusual prices separately for investigation.
df = df[
    df["price"].notna() &
    (df["price"] > 0) &
    df["date"].notna()
].copy()

df["price_under_1000_flag"] = df["price"] < 1000
df["price_over_10m_flag"] = df["price"] > 10_000_000

# Create initial modelling features
df["sale_year"] = df["date"].dt.year
df["sale_month"] = df["date"].dt.month

# Keep the target on its original GBP scale
feature_cols = [
    "sale_year", "sale_month", "property_type",
    "old_new", "duration", "district"
]

# Exclude rows missing essential categorical predictors for this
# initial experiment; report how many are removed.
before = len(df)
df = df.dropna(subset=feature_cols).copy()

print("\nRows before predictor completeness check:", before)
print("Rows after predictor completeness check:", len(df))
print("Rows excluded:", before - len(df))
print("Remaining missing values:")
display(df[feature_cols + ["price"]].isna().sum())

# Save for the next stage
out = Path("../data/processed")
out.mkdir(parents=True, exist_ok=True)
df.to_csv(out / "greater_london_clean.csv", index=False)

print("\nSaved: data/processed/greater_london_clean.csv")


Prices below £1,000: 99
Prices below £10,000: 1030
Prices above £10 million: 1414
Prices above £50 million: 126

Lowest 20 transactions:


,price,date,property_type,postcode,district
1018419,1,2024-08-20,F,N7 0AA,ISLINGTON
2116124,1,2025-04-01,F,SW18 2HG,WANDSWORTH
2116125,1,2025-04-01,F,SW18 2HG,WANDSWORTH
2116126,1,2025-04-01,F,SW18 2HG,WANDSWORTH
2116127,1,2025-04-01,F,SW18 2HG,WANDSWORTH
523,100,2023-02-16,O,RM12 4TQ,HAVERING
16362,100,2023-05-22,O,E3 4SF,TOWER HAMLETS
90544,100,2023-07-21,O,RM8 1YX,BARKING AND DAGENHAM
109956,100,2023-08-01,O,CR2 7PB,CROYDON
239148,100,2023-12-21,O,BR2 0TY,BROMLEY



Highest 20 transactions:


,price,date,property_type,postcode,district
1930246,793020000,2025-09-30,O,WC2R 2PR,CITY OF WESTMINSTER
218797,267500000,2023-06-01,O,W1S 2YF,CITY OF WESTMINSTER
220784,267500000,2023-06-01,O,W1S 2YF,CITY OF WESTMINSTER
243404,267500000,2023-06-01,O,W1S 2RE,CITY OF WESTMINSTER
1866027,260000000,2025-10-31,O,W1D 6QF,CITY OF WESTMINSTER
834749,251000000,2023-08-04,O,EC2M 4AA,TOWER HAMLETS
153352,249500000,2023-09-28,O,EC1V 4EX,ISLINGTON
790312,201146664,2023-03-20,O,CR0 2WF,CROYDON
1864865,190498163,2025-02-28,O,<NA>,WANDSWORTH
1030970,180000000,2024-10-30,O,WC1V 6XX,CAMDEN



Rows before predictor completeness check: 311469
Rows after predictor completeness check: 311469
Rows excluded: 0
Remaining missing values:


sale_year        0
sale_month       0
property_type    0
old_new          0
duration         0
district         0
price            0
dtype: int64


Saved: data/processed/greater_london_clean.csv


In [7]:

# Investigate price distribution by property type

property_analysis = (
    df.groupby("property_type")
    .agg(
        transactions=("price", "size"),
        median_price=("price", "median"),
        mean_price=("price", "mean"),
        minimum_price=("price", "min"),
        maximum_price=("price", "max"),
        below_1000=("price", lambda s: (s < 1000).sum()),
        below_10000=("price", lambda s: (s < 10000).sum()),
        above_10m=("price", lambda s: (s > 10_000_000).sum())
    )
    .sort_values("transactions", ascending=False)
)

display(property_analysis.round(2))


,transactions,median_price,mean_price,minimum_price,maximum_price,below_1000,below_10000,above_10m
property_type,,,,,,,,
F,160371,430000.0,584688.25,1,74933000,5,6,234
T,78802,565000.0,795717.39,39000,82000000,0,0,113
S,41540,610000.0,789693.01,77000,35750000,0,0,49
O,17601,513482.0,2629774.56,100,793020000,94,1024,940
D,13155,900000.0,1329472.87,46500,65000000,0,0,78


In [13]:

# Create an initial residential-only dataset

residential = df[
    df["property_type"].isin(["F", "T", "S", "D"])
].copy()

print("Transactions before filtering:", len(df))
print("Residential transactions:", len(residential))
print("Excluded 'O' transactions:", len(df) - len(residential))

# Flag suspiciously low prices for review
print("\nResidential sales below £1,000:")
display(
    residential.loc[
        residential["price"] < 1000,
        ["price", "date", "property_type", "ppd_category",
         "record_status", "district"]
    ]
)

print("\nResidential sales below £10,000:")
display(
    residential.loc[
        residential["price"] < 10000,
        ["price", "date", "property_type", "ppd_category",
         "record_status", "district"]
    ]
)

# Summarise the candidate modelling population
print("\nResidential price distribution:")
display(
    residential["price"].describe(
        percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]
    )
)

print("\nCounts by year and property type:")
display(
    pd.crosstab(
        residential["sale_year"],
        residential["property_type"]
    )
)

# Save the candidate dataset without removing price extremes
output_path = Path(
    "../data/processed/greater_london_residential_candidate.csv"
)
residential.to_csv(output_path, index=False)

print("\nSaved:", output_path)


Transactions before filtering: 311469
Residential transactions: 293868
Excluded 'O' transactions: 17601

Residential sales below £1,000:


,price,date,property_type,ppd_category,record_status,district
1018419,1,2024-08-20,F,A,A,ISLINGTON
2116124,1,2025-04-01,F,A,A,WANDSWORTH
2116125,1,2025-04-01,F,A,A,WANDSWORTH
2116126,1,2025-04-01,F,A,A,WANDSWORTH
2116127,1,2025-04-01,F,A,A,WANDSWORTH



Residential sales below £10,000:


,price,date,property_type,ppd_category,record_status,district
1018419,1,2024-08-20,F,A,A,ISLINGTON
2116124,1,2025-04-01,F,A,A,WANDSWORTH
2116125,1,2025-04-01,F,A,A,WANDSWORTH
2116126,1,2025-04-01,F,A,A,WANDSWORTH
2116127,1,2025-04-01,F,A,A,WANDSWORTH
2188602,8000,2025-10-27,F,A,A,EALING



Residential price distribution:


count    2.938680e+05
mean     7.035956e+05
std      1.023457e+06
min      1.000000e+00
1%       1.400000e+05
25%      3.750000e+05
50%      5.100000e+05
75%      7.400000e+05
99%      3.750000e+06
max      8.200000e+07
Name: price, dtype: float64


Counts by year and property type:


property_type,D,F,S,T
sale_year,,,,
2023,4034,51319,12154,23205
2024,4469,54820,14226,27008
2025,4652,54232,15160,28589



Saved: ..\data\processed\greater_london_residential_candidate.csv


In [14]:

# Finalise the initial dataset and create chronological splits

from pathlib import Path
import pandas as pd

# Copy the residential candidate dataset
model_data = residential.copy()

# Documented initial rule: exclude transactions below £1,000
# because they are implausible as ordinary residential sale prices.
before = len(model_data)
model_data = model_data[model_data["price"] >= 1000].copy()

print("Rows before low-price filter:", before)
print("Rows after low-price filter:", len(model_data))
print("Rows removed:", before - len(model_data))

# Confirm years and transaction counts
model_data["sale_year"] = model_data["date"].dt.year

print("\nRows by year:")
display(model_data.groupby("sale_year").size().rename("transactions"))

# Chronological split
train = model_data[model_data["sale_year"] == 2023].copy()
validation = model_data[model_data["sale_year"] == 2024].copy()
test = model_data[model_data["sale_year"] == 2025].copy()

print("\nSplit sizes:")
print("Train (2023):", len(train))
print("Validation (2024):", len(validation))
print("Test (2025):", len(test))

# Save datasets
out = Path("../data/processed")
out.mkdir(parents=True, exist_ok=True)

model_data.to_csv(out / "greater_london_model_data.csv", index=False)
train.to_csv(out / "train_2023.csv", index=False)
validation.to_csv(out / "validation_2024.csv", index=False)
test.to_csv(out / "test_2025.csv", index=False)

print("\nSaved all datasets to data/processed/")


Rows before low-price filter: 293868
Rows after low-price filter: 293863
Rows removed: 5

Rows by year:


sale_year
2023     90712
2024    100522
2025    102629
Name: transactions, dtype: int64


Split sizes:
Train (2023): 90712
Validation (2024): 100522
Test (2025): 102629

Saved all datasets to data/processed/


In [15]:
OUT = Path("../data/processed")

files = {
    "train_2023.csv": 2023,
    "validation_2024.csv": 2024,
    "test_2025.csv": 2025,
}

datasets = {}
all_passed = True

for filename, expected_year in files.items():
    path = OUT / filename

    if not path.exists():
        print(f"FAIL: Missing file: {path}")
        all_passed = False
        continue

    data = pd.read_csv(
        path,
        dtype={"transaction_id": "string"},
        low_memory=False
    )
    datasets[filename] = data

    dates = pd.to_datetime(data["date"], errors="coerce")
    ids = data["transaction_id"].str.strip()

    checks = {
        "Correct year": dates.dt.year.eq(expected_year).all(),
        "No missing prices": data["price"].notna().all(),
        "All prices >= £1,000": (data["price"] >= 1000).all(),
        "No property type O": data["property_type"].ne("O").all(),
        "No missing transaction IDs": ids.notna().all() & ids.ne("").all(),
        "No duplicate transaction IDs": not ids.duplicated().any(),
    }

    print(f"\n--- {filename} ---")
    print("Rows:", len(data))
    print("Date range:", dates.min(), "to", dates.max())

    for name, passed in checks.items():
        print(f"{'PASS' if passed else 'FAIL'}: {name}")
        if not passed:
            all_passed = False

# Check for overlap between splits
if len(datasets) == 3:
    id_sets = {
        name: set(data["transaction_id"].dropna().str.strip())
        for name, data in datasets.items()
    }

    names = list(id_sets)
    print("\n--- CROSS-SPLIT CHECKS ---")

    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            overlap = id_sets[names[i]] & id_sets[names[j]]
            passed = len(overlap) == 0
            print(
                f"{'PASS' if passed else 'FAIL'}: "
                f"{names[i]} vs {names[j]} — "
                f"{len(overlap)} overlapping IDs"
            )
            if not passed:
                all_passed = False

print("\nFINAL STATUS:",
      "ALL CHECKS PASSED" if all_passed else "REVIEW FAILURES")


--- train_2023.csv ---
Rows: 90712
Date range: 2023-01-01 00:00:00 to 2023-12-29 00:00:00
PASS: Correct year
PASS: No missing prices
PASS: All prices >= £1,000
PASS: No property type O
PASS: No missing transaction IDs
PASS: No duplicate transaction IDs

--- validation_2024.csv ---
Rows: 100522
Date range: 2024-01-01 00:00:00 to 2024-12-31 00:00:00
PASS: Correct year
PASS: No missing prices
PASS: All prices >= £1,000
PASS: No property type O
PASS: No missing transaction IDs
PASS: No duplicate transaction IDs

--- test_2025.csv ---
Rows: 102629
Date range: 2025-01-01 00:00:00 to 2025-12-31 00:00:00
PASS: Correct year
PASS: No missing prices
PASS: All prices >= £1,000
PASS: No property type O
PASS: No missing transaction IDs
PASS: No duplicate transaction IDs

--- CROSS-SPLIT CHECKS ---
PASS: train_2023.csv vs validation_2024.csv — 0 overlapping IDs
PASS: train_2023.csv vs test_2025.csv — 0 overlapping IDs
PASS: validation_2024.csv vs test_2025.csv — 0 overlapping IDs

FINAL STATUS: ALL 


## Final Data Cleaning and Validation Summary

### Geographic and temporal scope
- Study area: Greater London, England.
- Source: HM Land Registry Price Paid data for 2023, 2024 and 2025.
- Initial regional dataset: 311,469 transactions.

### Cleaning decisions
- Excluded 17,601 records classified as property type `O`.
- Excluded five residential records priced below £1,000.
- Retained high-value transactions pending model evaluation.
- Kept the original source files unchanged.

### Final datasets
- Training (2023): 90,712 transactions.
- Validation (2024): 100,522 transactions.
- Testing (2025): 102,629 transactions.
- Total: 293,863 transactions.

### Validation
- All saved files passed year, price, property-type and identifier checks.
- No duplicate transaction IDs within individual splits.
- No transaction ID overlap between splits.

### Limitations
- EPC features have not yet been integrated.
- The £1,000 threshold is a provisional filtering rule.
- The excluded `O` category may contain some relevant residential transactions.
- The dataset covers three years and one English region.
